# Air Pollution Data Preparation

This notebook prepares real-world air quality measurements for five major Indian cities: Bengaluru, Chennai, Delhi, Kolkata, and Mumbai.

The air pollution data were obtained from the OpenAQ historical archive and cover the period from January 2020 to December 2022.

The preparation process includes:

- Combining yearly raw measurement files
- Validating pollutant measurements and units
- Converting sub-daily observations into daily averages
- Aggregating monitoring locations to city-level observations
- Evaluating temporal coverage and missing values
- Preparing the pollution dataset for integration with meteorological data

The pollutants considered are:

- PM2.5
- PM10
- NO2
- SO2
- O3
- CO

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os

Mounted at /content/drive


In [ ]:
base_path = "/content/drive/MyDrive/Air_Pollution_Project/Raw_Data"

file_paths = [
    os.path.join(base_path, "openaq_2020_raw_selected.csv"),
    os.path.join(base_path, "openaq_2021_raw_selected.csv"),
    os.path.join(base_path, "openaq_2022_raw_selected.csv")
]

pollution_parts = []

for file in file_paths:

    temp = pd.read_csv(file)

    print(
        os.path.basename(file),
        "→",
        f"{len(temp):,} rows"
    )

    pollution_parts.append(temp)

pollution_raw = pd.concat(
    pollution_parts,
    ignore_index=True
)

print("\nCombined shape:", pollution_raw.shape)

display(pollution_raw.head())

openaq_2020_raw_selected.csv → 4,360,414 rows
openaq_2021_raw_selected.csv → 1,915,342 rows
openaq_2022_raw_selected.csv → 781,802 rows

Combined shape: (7057558, 6)


,location_id,datetime,parameter,units,value,City
0,5547,2020-12-01 19:00:00+00:00,pm25,µg/m³,28.50,Bengaluru
1,5547,2020-12-01 19:15:00+00:00,pm25,µg/m³,23.65,Bengaluru
2,5547,2020-12-01 19:45:00+00:00,pm25,µg/m³,19.59,Bengaluru
3,5547,2020-12-01 21:00:00+00:00,pm25,µg/m³,15.94,Bengaluru
4,5547,2020-12-01 22:30:00+00:00,pm25,µg/m³,21.06,Bengaluru


In [ ]:
print("Dataset shape:")
print(pollution_raw.shape)

print("\nColumns:")
print(pollution_raw.columns.tolist())

print("\nData types:")
print(pollution_raw.dtypes)

print("\nMissing values:")
print(pollution_raw.isnull().sum())

print("\nPollutants:")
print(pollution_raw["parameter"].value_counts())

print("\nCities:")
print(pollution_raw["City"].value_counts())

Dataset shape:
(7057558, 6)

Columns:
['location_id', 'datetime', 'parameter', 'units', 'value', 'City']

Data types:
location_id      int64
datetime        object
parameter       object
units           object
value          float64
City            object
dtype: object

Missing values:
location_id    0
datetime       0
parameter      0
units          0
value          0
City           0
dtype: int64

Pollutants:
parameter
pm25    1298673
no2     1222772
co      1180594
o3      1157406
pm10    1114452
so2     1083661
Name: count, dtype: int64

Cities:
City
Delhi        3812950
Mumbai       1231682
Kolkata       929540
Bengaluru     676382
Chennai       407004
Name: count, dtype: int64


## Data Validation

Before aggregating the raw monitoring-station measurements, the dataset is validated for date coverage, measurement units, and potentially invalid pollutant values.

This step helps ensure that only meaningful and comparable observations are used to construct the daily city-level air pollution dataset.

In [ ]:
# Convert datetime to pandas datetime format
pollution_raw["datetime"] = pd.to_datetime(
    pollution_raw["datetime"],
    utc=True
)

print("Earliest observation:")
print(pollution_raw["datetime"].min())

print("\nLatest observation:")
print(pollution_raw["datetime"].max())

print("\nObservations by year:")
print(
    pollution_raw["datetime"]
    .dt.year
    .value_counts()
    .sort_index()
)

Earliest observation:
2020-01-01 00:00:00+00:00

Latest observation:
2022-12-31 18:30:00+00:00

Observations by year:
datetime
2020    4360414
2021    1915342
2022     781802
Name: count, dtype: int64


In [ ]:
# Check measurement units used for each pollutant

unit_check = (
    pollution_raw
    .groupby("parameter")["units"]
    .value_counts()
)

print(unit_check)

parameter  units
co         µg/m³    1180594
no2        µg/m³    1222772
o3         µg/m³    1157043
           ppm          363
pm10       µg/m³    1114452
pm25       µg/m³    1298673
so2        µg/m³    1083661
Name: count, dtype: int64


In [ ]:
# Summary statistics for pollutant measurements

pollutant_summary = (
    pollution_raw
    .groupby("parameter")["value"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .round(2)
)

display(pollutant_summary)

,count,mean,median,std,min,max
parameter,,,,,,
co,1180594,1.779267e+20,790.00,1.933265e+23,-4.889000e+08,2.100592e+26
no2,1222772,3.440000e+01,23.56,4.847000e+01,-9.999000e+02,1.778750e+04
o3,1157406,3.571000e+01,19.10,4.158400e+02,-1.436000e+04,8.778851e+04
pm10,1114452,1.717200e+02,125.00,5.246400e+02,-9.999000e+03,4.764784e+05
pm25,1298673,8.691000e+01,54.00,1.669300e+02,-2.574000e+03,9.999900e+03
so2,1083661,1.385000e+01,9.06,8.702000e+01,-1.131900e+02,4.692598e+04


In [ ]:
# Check potentially invalid measurements

validation_summary = []

for pollutant in sorted(pollution_raw["parameter"].unique()):

    subset = pollution_raw[
        pollution_raw["parameter"] == pollutant
    ]["value"]

    validation_summary.append({
        "Pollutant": pollutant,
        "Total": len(subset),
        "Negative": (subset < 0).sum(),
        "Zero": (subset == 0).sum(),
        "Positive": (subset > 0).sum()
    })

validation_summary = pd.DataFrame(validation_summary)

display(validation_summary)

,Pollutant,Total,Negative,Zero,Positive
0,co,1180594,1717,24079,1154798
1,no2,1222772,2237,24772,1195763
2,o3,1157406,1140,26624,1129642
3,pm10,1114452,1626,16761,1096065
4,pm25,1298673,7233,13677,1277763
5,so2,1083661,732,22414,1060515


## Cleaning Invalid Pollution Measurements

Raw monitoring data may contain invalid negative concentrations, error-coded values, inconsistent measurement units, and extreme instrument-related readings.

Before daily aggregation:

- Negative pollutant concentrations are treated as invalid and removed.
- Measurements recorded in inconsistent units are excluded to maintain unit consistency.
- Extreme physically implausible values are identified using pollutant-specific validation limits rather than automatically removing statistical outliers.
- Zero measurements are retained at this stage because zero concentrations are not necessarily data-entry errors.

This approach preserves potentially genuine high-pollution episodes while removing clearly invalid monitoring records.

In [ ]:
# Examine pollutant distributions after removing
# negative values and inconsistent units

temp_valid = pollution_raw[
    (pollution_raw["value"] >= 0) &
    (pollution_raw["units"] == "µg/m³")
].copy()

percentile_summary = (
    temp_valid
    .groupby("parameter")["value"]
    .quantile([
        0.50,
        0.90,
        0.95,
        0.99,
        0.999,
        0.9999,
        1.00
    ])
    .unstack()
)

percentile_summary.columns = [
    "50%",
    "90%",
    "95%",
    "99%",
    "99.9%",
    "99.99%",
    "Max"
]

display(
    percentile_summary.round(2)
)

,50%,90%,95%,99%,99.9%,99.99%,Max
parameter,,,,,,,
co,790.00,2160.00,2940.00,5660.00,10000.00,14550.12,2.100592e+26
no2,23.60,76.30,100.01,161.06,283.30,999.99,1.778750e+04
o3,19.14,78.05,104.84,168.10,658.97,3787.78,8.778851e+04
pm10,125.00,354.00,464.00,862.82,1450.74,9999.90,4.764784e+05
pm25,54.87,191.00,268.00,613.00,1000.00,7047.77,9.999900e+03
so2,9.08,24.70,34.45,74.51,382.65,1525.58,4.692598e+04


## Treatment of Invalid and Extreme Measurements

The raw monitoring data contained several data-quality issues, including negative pollutant concentrations, inconsistent measurement units, and extremely large values likely caused by sensor or recording errors.

The following cleaning rules were applied before aggregation:

- Negative concentration values were removed because pollutant concentrations cannot physically be negative.
- Measurements were restricted to µg/m³ to maintain consistent units across observations.
- Extreme upper-tail observations above the 99.9th percentile for each pollutant were excluded as a conservative quality-control step.
- Zero values were retained because they may represent valid low-concentration measurements.

This cleaning was performed on raw monitoring observations before calculating daily averages, preventing erroneous sensor readings from disproportionately affecting city-level pollution estimates.

In [ ]:
# Start with a copy
pollution_clean = pollution_raw.copy()

original_rows = len(pollution_clean)

# 1. Keep only non-negative measurements
pollution_clean = pollution_clean[
    pollution_clean["value"] >= 0
].copy()

# 2. Keep consistent measurement units
pollution_clean = pollution_clean[
    pollution_clean["units"] == "µg/m³"
].copy()

# 3. Calculate pollutant-specific 99.9th percentile limits
upper_limits = (
    pollution_clean
    .groupby("parameter")["value"]
    .quantile(0.999)
)

print("Upper cleaning limits:")
print(upper_limits.round(2))

# 4. Remove values above each pollutant's limit
pollution_clean = pollution_clean[
    pollution_clean.apply(
        lambda row:
        row["value"] <= upper_limits[row["parameter"]],
        axis=1
    )
].copy()

removed_rows = original_rows - len(pollution_clean)

print("\nOriginal rows:", f"{original_rows:,}")
print("Rows remaining:", f"{len(pollution_clean):,}")
print("Rows removed:", f"{removed_rows:,}")

print(
    "Percentage removed:",
    round(removed_rows / original_rows * 100, 3),
    "%"
)

# Final cleaned ranges
clean_summary = (
    pollution_clean
    .groupby("parameter")["value"]
    .agg(["count", "min", "median", "mean", "max"])
    .round(2)
)

display(clean_summary)

Upper cleaning limits:
parameter
co      10000.00
no2       283.30
o3        658.97
pm10     1450.74
pm25     1000.00
so2       382.65
Name: value, dtype: float64

Original rows: 7,057,558
Rows remaining: 7,036,747
Rows removed: 20,811
Percentage removed: 0.295 %


,count,min,median,mean,max
parameter,,,,,
co,1178488,0.0,790.00,1047.99,10000.00
no2,1219315,0.0,23.60,33.94,283.30
o3,1154747,0.0,19.10,31.95,658.96
pm10,1111713,0.0,125.00,168.35,1450.70
pm25,1290638,0.0,54.76,88.31,1000.00
so2,1081846,0.0,9.05,12.86,382.65


## Daily City-Level Aggregation

The raw OpenAQ dataset contains multiple measurements per day from multiple monitoring locations within each city.

To create a consistent dataset for analysis and prediction, aggregation was performed in two stages:

1. Sub-daily measurements were averaged within each monitoring location for each pollutant and date.
2. Daily monitoring-location averages were then averaged across available locations within each city.

This hierarchical approach prevents monitoring locations with more frequent measurements from receiving disproportionate weight in the city-level daily averages.

The resulting dataset contains one observation for each city and date, with separate columns for each pollutant.

In [ ]:
# Convert UTC timestamps to Indian Standard Time
pollution_clean["datetime_ist"] = (
    pollution_clean["datetime"]
    .dt.tz_convert("Asia/Kolkata")
)

# Extract local calendar date
pollution_clean["Date"] = (
    pollution_clean["datetime_ist"]
    .dt.date
)

pollution_clean["Date"] = pd.to_datetime(
    pollution_clean["Date"]
)

print(
    "Date range:",
    pollution_clean["Date"].min(),
    "to",
    pollution_clean["Date"].max()
)

display(
    pollution_clean[
        [
            "datetime",
            "datetime_ist",
            "Date",
            "City",
            "parameter",
            "value"
        ]
    ].head()
)

Date range: 2020-01-01 00:00:00 to 2023-01-01 00:00:00


,datetime,datetime_ist,Date,City,parameter,value
0,2020-12-01 19:00:00+00:00,2020-12-02 00:30:00+05:30,2020-12-02,Bengaluru,pm25,28.50
1,2020-12-01 19:15:00+00:00,2020-12-02 00:45:00+05:30,2020-12-02,Bengaluru,pm25,23.65
2,2020-12-01 19:45:00+00:00,2020-12-02 01:15:00+05:30,2020-12-02,Bengaluru,pm25,19.59
3,2020-12-01 21:00:00+00:00,2020-12-02 02:30:00+05:30,2020-12-02,Bengaluru,pm25,15.94
4,2020-12-01 22:30:00+00:00,2020-12-02 04:00:00+05:30,2020-12-02,Bengaluru,pm25,21.06


In [ ]:
location_daily = (
    pollution_clean
    .groupby(
        [
            "City",
            "location_id",
            "Date",
            "parameter"
        ],
        as_index=False
    )["value"]
    .mean()
)

print(
    "Location-level daily records:",
    f"{len(location_daily):,}"
)

display(location_daily.head())

Location-level daily records: 272,462


,City,location_id,Date,parameter,value
0,Bengaluru,5547,2020-01-02,co,3325.185185
1,Bengaluru,5547,2020-01-02,no2,55.897407
2,Bengaluru,5547,2020-01-02,o3,5.443333
3,Bengaluru,5547,2020-01-02,pm25,24.917037
4,Bengaluru,5547,2020-01-02,so2,3.782963


In [ ]:
# Aggregate monitoring locations to city-level daily values
# Median is used instead of mean to reduce the influence of
# malfunctioning or anomalous individual monitoring stations.

city_daily_long = (
    location_daily
    .groupby(
        [
            "City",
            "Date",
            "parameter"
        ],
        as_index=False
    )["value"]
    .median()
)

print(
    "City-level pollutant records:",
    f"{len(city_daily_long):,}"
)

display(city_daily_long.head())

City-level pollutant records: 23,295


,City,Date,parameter,value
0,Bengaluru,2020-01-01,co,247.754587
1,Bengaluru,2020-01-01,no2,31.506189
2,Bengaluru,2020-01-01,o3,30.566349
3,Bengaluru,2020-01-01,pm10,66.219618
4,Bengaluru,2020-01-01,pm25,26.703125


In [ ]:
pollution_daily = (
    city_daily_long
    .pivot(
        index=["Date", "City"],
        columns="parameter",
        values="value"
    )
    .reset_index()
)

pollution_daily.columns.name = None

# Rename pollutants professionally
pollution_daily = pollution_daily.rename(
    columns={
        "pm25": "PM2.5",
        "pm10": "PM10",
        "no2": "NO2",
        "so2": "SO2",
        "o3": "O3",
        "co": "CO"
    }
)

pollution_daily = pollution_daily[
    [
        "Date",
        "City",
        "PM2.5",
        "PM10",
        "NO2",
        "SO2",
        "O3",
        "CO"
    ]
]

pollution_daily = pollution_daily.sort_values(
    ["City", "Date"]
).reset_index(drop=True)

print("Final pollution dataset shape:")
print(pollution_daily.shape)

display(pollution_daily.head(10))

Final pollution dataset shape:
(4439, 8)


,Date,City,PM2.5,PM10,NO2,SO2,O3,CO
0,2020-01-01,Bengaluru,26.703125,66.219618,31.506189,6.228033,30.566349,247.754587
1,2020-01-02,Bengaluru,24.315661,62.571429,28.996195,5.079903,21.913704,864.569079
2,2020-01-03,Bengaluru,29.665271,72.838235,29.614459,5.535566,30.127463,668.727273
3,2020-01-04,Bengaluru,55.235786,114.010870,35.472778,6.702294,38.663864,1069.550189
4,2020-01-05,Bengaluru,51.294156,101.563218,30.340575,8.320714,43.419048,688.467578
5,2020-01-06,Bengaluru,22.642570,62.554217,25.856250,5.966067,22.645114,666.959270
6,2020-01-07,Bengaluru,24.838682,61.011628,28.709500,6.703654,29.825634,649.030899
7,2020-01-08,Bengaluru,31.501083,79.282609,33.996337,4.558495,46.321798,800.207985
8,2020-01-09,Bengaluru,41.306190,91.720930,38.944933,6.294483,32.650000,692.804878
9,2020-01-10,Bengaluru,38.605655,89.869048,38.536590,7.940110,33.272222,785.418789


In [ ]:
expected_days = pd.date_range(
    "2020-01-01",
    "2022-12-31",
    freq="D"
)

print("Expected days per city:", len(expected_days))
print("Expected total City-Date rows:", len(expected_days) * 5)

print("\nActual unique dates by city:")

coverage = (
    pollution_daily
    .groupby("City")["Date"]
    .nunique()
)

print(coverage)

print("\nMissing values by pollutant:")

print(
    pollution_daily[
        ["PM2.5", "PM10", "NO2", "SO2", "O3", "CO"]
    ]
    .isnull()
    .sum()
)

print("\nNon-missing daily observations by City × Pollutant:")

coverage_table = (
    pollution_daily
    .groupby("City")[
        ["PM2.5", "PM10", "NO2", "SO2", "O3", "CO"]
    ]
    .count()
)

display(coverage_table)

Expected days per city: 1096
Expected total City-Date rows: 5480

Actual unique dates by city:
City
Bengaluru    688
Chennai      974
Delhi        967
Kolkata      836
Mumbai       974
Name: Date, dtype: int64

Missing values by pollutant:
PM2.5      6
PM10     727
NO2      652
SO2      655
O3       650
CO       649
dtype: int64

Non-missing daily observations by City × Pollutant:


,PM2.5,PM10,NO2,SO2,O3,CO
City,,,,,,
Bengaluru,684,625,683,684,684,686
Chennai,974,757,775,775,771,773
Delhi,966,781,780,779,785,782
Kolkata,836,763,763,763,763,763
Mumbai,973,786,786,783,786,786



## Temporal Coverage Assessment

A complete city-date grid was created for all five cities from January 1, 2020 to December 31, 2022.

This ensures that days with no available pollution measurements are explicitly represented as missing observations rather than being absent from the dataset.

Coverage was then evaluated separately for each pollutant and city before deciding how missing values should be handled.

In [ ]:
# Create complete City × Date grid

cities = [
    "Bengaluru",
    "Chennai",
    "Delhi",
    "Kolkata",
    "Mumbai"
]

all_dates = pd.date_range(
    start="2020-01-01",
    end="2022-12-31",
    freq="D"
)

complete_grid = pd.MultiIndex.from_product(
    [all_dates, cities],
    names=["Date", "City"]
).to_frame(index=False)

pollution_complete = complete_grid.merge(
    pollution_daily,
    on=["Date", "City"],
    how="left"
)

pollution_complete = pollution_complete.sort_values(
    ["City", "Date"]
).reset_index(drop=True)

print("Shape:", pollution_complete.shape)

print(
    "Duplicate City-Date rows:",
    pollution_complete.duplicated(
        ["Date", "City"]
    ).sum()
)

display(pollution_complete.head())

Shape: (5480, 8)
Duplicate City-Date rows: 0


,Date,City,PM2.5,PM10,NO2,SO2,O3,CO
0,2020-01-01,Bengaluru,26.703125,66.219618,31.506189,6.228033,30.566349,247.754587
1,2020-01-02,Bengaluru,24.315661,62.571429,28.996195,5.079903,21.913704,864.569079
2,2020-01-03,Bengaluru,29.665271,72.838235,29.614459,5.535566,30.127463,668.727273
3,2020-01-04,Bengaluru,55.235786,114.010870,35.472778,6.702294,38.663864,1069.550189
4,2020-01-05,Bengaluru,51.294156,101.563218,30.340575,8.320714,43.419048,688.467578


In [ ]:
pollutants = [
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "O3",
    "CO"
]

true_coverage = (
    pollution_complete
    .groupby("City")[pollutants]
    .count()
)

true_coverage_pct = (
    true_coverage / 1096 * 100
).round(1)

print("Available days:")
display(true_coverage)

print("\nCoverage percentage:")
display(true_coverage_pct)

Available days:


,PM2.5,PM10,NO2,SO2,O3,CO
City,,,,,,
Bengaluru,684,625,683,684,684,686
Chennai,973,757,775,775,771,773
Delhi,965,781,780,779,785,782
Kolkata,836,763,763,763,763,763
Mumbai,972,786,786,783,786,786



Coverage percentage:


,PM2.5,PM10,NO2,SO2,O3,CO
City,,,,,,
Bengaluru,62.4,57.0,62.3,62.4,62.4,62.6
Chennai,88.8,69.1,70.7,70.7,70.3,70.5
Delhi,88.0,71.3,71.2,71.1,71.6,71.4
Kolkata,76.3,69.6,69.6,69.6,69.6,69.6
Mumbai,88.7,71.7,71.7,71.4,71.7,71.7


In [ ]:
# Add year for coverage analysis
pollution_complete["Year"] = pollution_complete["Date"].dt.year

pollutants = [
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "O3",
    "CO"
]

yearly_available = (
    pollution_complete
    .groupby(["City", "Year"])[pollutants]
    .count()
)

display(yearly_available)

PM2.5  PM10  NO2  SO2   O3   CO
City      Year                                 
Bengaluru 2020    361   332  362  362  360  362
          2021    214   184  212  213  215  215
          2022    109   109  109  109  109  109
Chennai   2020    363   347  361  361  360  360
          2021    352   275  279  279  276  278
          2022    258   135  135  135  135  135
Delhi     2020    363   362  362  363  364  363
          2021    349   285  285  284  286  285
          2022    253   134  133  132  135  134
Kolkata   2020    362   362  362  362  362  362
          2021    276   274  274  274  274  274
          2022    198   127  127  127  127  127
Mumbai    2020    362   361  361  361  361  361
          2021    351   289  289  287  289  289
          2022    259   136  136  135  136  136

In [ ]:
days_per_year = {
    2020: 366,
    2021: 365,
    2022: 365
}

yearly_coverage_pct = yearly_available.copy()

for idx in yearly_coverage_pct.index:

    year = idx[1]

    yearly_coverage_pct.loc[idx] = (
        yearly_coverage_pct.loc[idx]
        / days_per_year[year]
        * 100
    )

yearly_coverage_pct = yearly_coverage_pct.round(1)

print("Coverage percentage by City × Year:")

display(yearly_coverage_pct)

Coverage percentage by City × Year:


/tmp/ipykernel_2600/1266593561.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '98.63387978142076' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  yearly_coverage_pct.loc[idx] = (
/tmp/ipykernel_2600/1266593561.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '90.7103825136612' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  yearly_coverage_pct.loc[idx] = (
/tmp/ipykernel_2600/1266593561.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '98.90710382513662' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  yearly_coverage_pct.loc[idx] = (
/tmp/ipykernel_2600/1266593561.py:13: FutureWarning: Setting an item of incom

PM2.5  PM10   NO2   SO2    O3    CO
City      Year                                     
Bengaluru 2020   98.6  90.7  98.9  98.9  98.4  98.9
          2021   58.6  50.4  58.1  58.4  58.9  58.9
          2022   29.9  29.9  29.9  29.9  29.9  29.9
Chennai   2020   99.2  94.8  98.6  98.6  98.4  98.4
          2021   96.4  75.3  76.4  76.4  75.6  76.2
          2022   70.7  37.0  37.0  37.0  37.0  37.0
Delhi     2020   99.2  98.9  98.9  99.2  99.5  99.2
          2021   95.6  78.1  78.1  77.8  78.4  78.1
          2022   69.3  36.7  36.4  36.2  37.0  36.7
Kolkata   2020   98.9  98.9  98.9  98.9  98.9  98.9
          2021   75.6  75.1  75.1  75.1  75.1  75.1
          2022   54.2  34.8  34.8  34.8  34.8  34.8
Mumbai    2020   98.9  98.6  98.6  98.6  98.6  98.6
          2021   96.2  79.2  79.2  78.6  79.2  79.2
          2022   71.0  37.3  37.3  37.0  37.3  37.3

In [ ]:
pm25_coverage = (
    yearly_coverage_pct[["PM2.5"]]
    .reset_index()
    .pivot(
        index="City",
        columns="Year",
        values="PM2.5"
    )
)

print("PM2.5 daily coverage (%)")

display(pm25_coverage)

PM2.5 daily coverage (%)


Year,2020,2021,2022
City,,,
Bengaluru,98.6,58.6,29.9
Chennai,99.2,96.4,70.7
Delhi,99.2,95.6,69.3
Kolkata,98.9,75.6,54.2
Mumbai,98.9,96.2,71.0


In [ ]:
pollution_complete["All_Pollutants_Available"] = (
    pollution_complete[pollutants]
    .notna()
    .all(axis=1)
)

complete_case_summary = (
    pollution_complete
    .groupby(["City", "Year"])
    ["All_Pollutants_Available"]
    .agg(["sum", "count"])
)

complete_case_summary["Percentage"] = (
    complete_case_summary["sum"]
    / complete_case_summary["count"]
    * 100
).round(1)

complete_case_summary = complete_case_summary.rename(
    columns={
        "sum": "Complete_Days",
        "count": "Total_Days"
    }
)

display(complete_case_summary)

Complete_Days  Total_Days  Percentage
City      Year                                       
Bengaluru 2020            332         366        90.7
          2021            184         365        50.4
          2022            109         365        29.9
Chennai   2020            346         366        94.5
          2021            272         365        74.5
          2022            135         365        37.0
Delhi     2020            362         366        98.9
          2021            284         365        77.8
          2022            131         365        35.9
Kolkata   2020            362         366        98.9
          2021            274         365        75.1
          2022            127         365        34.8
Mumbai    2020            360         366        98.4
          2021            287         365        78.6
          2022            135         365        37.0

In [ ]:
# Keep 2020–2021 for primary analysis and modeling

pollution_model = pollution_complete[
    pollution_complete["Date"].between(
        "2020-01-01",
        "2021-12-31"
    )
].copy()

# Remove temporary analysis columns if present
columns_to_drop = [
    "Year",
    "All_Pollutants_Available"
]

pollution_model = pollution_model.drop(
    columns=[
        col for col in columns_to_drop
        if col in pollution_model.columns
    ]
)

pollution_model = pollution_model.sort_values(
    ["City", "Date"]
).reset_index(drop=True)

print("Dataset shape:")
print(pollution_model.shape)

print("\nDate range:")
print(
    pollution_model["Date"].min(),
    "to",
    pollution_model["Date"].max()
)

print("\nRows per city:")
print(
    pollution_model["City"].value_counts()
)

Dataset shape:
(3655, 8)

Date range:
2020-01-01 00:00:00 to 2021-12-31 00:00:00

Rows per city:
City
Bengaluru    731
Chennai      731
Delhi        731
Kolkata      731
Mumbai       731
Name: count, dtype: int64


In [ ]:
pollutants = [
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "O3",
    "CO"
]

missing_summary = pd.DataFrame({
    "Missing_Count":
        pollution_model[pollutants].isnull().sum(),

    "Missing_Percentage":
        (
            pollution_model[pollutants]
            .isnull()
            .mean() * 100
        ).round(2)
})

display(missing_summary)

,Missing_Count,Missing_Percentage
PM2.5,302,8.26
PM10,584,15.98
NO2,508,13.90
SO2,509,13.93
O3,508,13.90
CO,506,13.84


In [ ]:
# Missing values by city

missing_by_city = (
    pollution_model
    .groupby("City")[pollutants]
    .apply(lambda x: x.isnull().mean() * 100)
    .round(2)
)

display(missing_by_city)

,PM2.5,PM10,NO2,SO2,O3,CO
City,,,,,,
Bengaluru,21.34,29.41,21.48,21.34,21.34,21.07
Chennai,2.19,14.91,12.45,12.45,13.00,12.72
Delhi,2.60,11.49,11.49,11.49,11.08,11.35
Kolkata,12.72,13.00,13.00,13.00,13.00,13.00
Mumbai,2.46,11.08,11.08,11.35,11.08,11.08


In [ ]:
def longest_missing_gap(series):

    missing = series.isna()

    groups = (
        missing.ne(missing.shift())
        .cumsum()
    )

    gaps = (
        missing
        .groupby(groups)
        .sum()
    )

    return gaps.max()


gap_results = []

for city in pollution_model["City"].unique():

    city_data = pollution_model[
        pollution_model["City"] == city
    ].sort_values("Date")

    for pollutant in pollutants:

        gap_results.append({
            "City": city,
            "Pollutant": pollutant,
            "Longest_Missing_Gap_Days":
                longest_missing_gap(
                    city_data[pollutant]
                )
        })


gap_summary = pd.DataFrame(gap_results)

gap_table = gap_summary.pivot(
    index="City",
    columns="Pollutant",
    values="Longest_Missing_Gap_Days"
)

display(gap_table)

Pollutant,CO,NO2,O3,PM10,PM2.5,SO2
City,,,,,,
Bengaluru,30,30,30,38,30,30
Chennai,30,30,30,30,8,30
Delhi,30,30,30,30,8,30
Kolkata,30,30,30,30,30,30
Mumbai,30,30,30,30,9,30


In [ ]:
# Sort chronologically before interpolation
pollution_model = pollution_model.sort_values(
    ["City", "Date"]
).reset_index(drop=True)

# Store missing counts before interpolation
missing_before = pollution_model[pollutants].isna().sum()

# Interpolate only short internal gaps (maximum 3 consecutive days)
pollution_model[pollutants] = (
    pollution_model
    .groupby("City", group_keys=False)[pollutants]
    .apply(
        lambda group: group.interpolate(
            method="linear",
            limit=3,
            limit_area="inside"
        )
    )
)

# Missing counts after interpolation
missing_after = pollution_model[pollutants].isna().sum()

interpolation_summary = pd.DataFrame({
    "Missing_Before": missing_before,
    "Missing_After": missing_after,
    "Values_Interpolated": missing_before - missing_after
})

display(interpolation_summary)

,Missing_Before,Missing_After,Values_Interpolated
PM2.5,302,152,150
PM10,584,331,253
NO2,508,275,233
SO2,509,276,233
O3,508,274,234
CO,506,274,232


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving India_5_Cities_Air_Quality_Weather_2020_2024.csv to India_5_Cities_Air_Quality_Weather_2020_2024.csv


In [ ]:
import pandas as pd
import io

# Automatically use the file you just uploaded
filename = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("Loaded file:", filename)
print("Shape:", df.shape)

display(df.head())

Loaded file: India_5_Cities_Air_Quality_Weather_2020_2024.csv
Shape: (9135, 18)


,Date,City,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,AQI,AQI_Bucket,Temperature_Mean,Relative_Humidity_Mean,Precipitation,Wind_Speed_Max,Surface_Pressure_Mean
0,2020-01-01,Bengaluru,408.3,48.8,87.6,24.1,186.4,6.6,9.39,78.4,103.6,192.4,Moderate,21.6,80,1.9,18.5,915.0
1,2020-01-02,Bengaluru,270.4,80.1,147.9,138.6,243.6,5.5,0.82,94.9,33.7,420.4,Severe,22.4,77,1.0,15.1,916.0
2,2020-01-03,Bengaluru,174.5,25.7,157.4,102.9,201.3,24.7,3.81,93.0,2.8,342.1,Very Poor,23.0,73,0.0,14.0,915.0
3,2020-01-04,Bengaluru,172.6,536.3,7.6,112.9,35.0,0.3,0.59,58.3,58.3,340.6,Very Poor,23.4,70,0.0,13.2,913.6
4,2020-01-05,Bengaluru,134.8,84.2,108.0,135.8,90.1,7.5,0.45,14.2,113.6,311.6,Very Poor,23.3,73,0.2,16.6,913.4


In [ ]:
weather_columns = [
    "Date",
    "City",
    "Temperature_Mean",
    "Relative_Humidity_Mean",
    "Precipitation",
    "Wind_Speed_Max",
    "Surface_Pressure_Mean"
]

weather = df[weather_columns].copy()

weather["Date"] = pd.to_datetime(weather["Date"])

# Restrict to the selected modeling period
weather = weather[
    weather["Date"].between(
        "2020-01-01",
        "2021-12-31"
    )
].copy()

print("Weather dataset shape:")
print(weather.shape)

print("\nDate range:")
print(
    weather["Date"].min(),
    "to",
    weather["Date"].max()
)

print("\nDuplicate City-Date rows:")
print(
    weather.duplicated(
        ["Date", "City"]
    ).sum()
)

print("\nMissing values:")
print(weather.isnull().sum())

display(weather.head())

Weather dataset shape:
(3655, 7)

Date range:
2020-01-01 00:00:00 to 2021-12-31 00:00:00

Duplicate City-Date rows:
0

Missing values:
Date                      0
City                      0
Temperature_Mean          0
Relative_Humidity_Mean    0
Precipitation             0
Wind_Speed_Max            0
Surface_Pressure_Mean     0
dtype: int64


,Date,City,Temperature_Mean,Relative_Humidity_Mean,Precipitation,Wind_Speed_Max,Surface_Pressure_Mean
0,2020-01-01,Bengaluru,21.6,80,1.9,18.5,915.0
1,2020-01-02,Bengaluru,22.4,77,1.0,15.1,916.0
2,2020-01-03,Bengaluru,23.0,73,0.0,14.0,915.0
3,2020-01-04,Bengaluru,23.4,70,0.0,13.2,913.6
4,2020-01-05,Bengaluru,23.3,73,0.2,16.6,913.4


## Creation of the Integrated Air Pollution Dataset

The cleaned daily city-level OpenAQ pollution measurements were merged with the meteorological dataset using `Date` and `City` as common identifiers.

The resulting dataset combines six air-pollution indicators with five meteorological variables for five major Indian cities during 2020–2021.

This integrated dataset forms the basis for subsequent feature engineering, statistical analysis, and next-day PM2.5 prediction.

In [ ]:
# Merge cleaned pollution data with weather data

final_df = pollution_model.merge(
    weather,
    on=["Date", "City"],
    how="left"
)

# Arrange columns clearly
final_df = final_df[
    [
        "Date",
        "City",
        "PM2.5",
        "PM10",
        "NO2",
        "SO2",
        "O3",
        "CO",
        "Temperature_Mean",
        "Relative_Humidity_Mean",
        "Precipitation",
        "Wind_Speed_Max",
        "Surface_Pressure_Mean"
    ]
]

final_df = final_df.sort_values(
    ["City", "Date"]
).reset_index(drop=True)

print("Final merged dataset shape:")
print(final_df.shape)

print("\nDate range:")
print(
    final_df["Date"].min(),
    "to",
    final_df["Date"].max()
)

print("\nDuplicate City-Date rows:")
print(
    final_df.duplicated(
        ["Date", "City"]
    ).sum()
)

print("\nMissing values:")
print(final_df.isnull().sum())

display(final_df.head(10))

Final merged dataset shape:
(3655, 13)

Date range:
2020-01-01 00:00:00 to 2021-12-31 00:00:00

Duplicate City-Date rows:
0

Missing values:
Date                        0
City                        0
PM2.5                     152
PM10                      331
NO2                       275
SO2                       276
O3                        274
CO                        274
Temperature_Mean            0
Relative_Humidity_Mean      0
Precipitation               0
Wind_Speed_Max              0
Surface_Pressure_Mean       0
dtype: int64


,Date,City,PM2.5,PM10,NO2,SO2,O3,CO,Temperature_Mean,Relative_Humidity_Mean,Precipitation,Wind_Speed_Max,Surface_Pressure_Mean
0,2020-01-01,Bengaluru,26.703125,66.219618,31.506189,6.228033,30.566349,247.754587,21.6,80,1.9,18.5,915.0
1,2020-01-02,Bengaluru,24.315661,62.571429,28.996195,5.079903,21.913704,864.569079,22.4,77,1.0,15.1,916.0
2,2020-01-03,Bengaluru,29.665271,72.838235,29.614459,5.535566,30.127463,668.727273,23.0,73,0.0,14.0,915.0
3,2020-01-04,Bengaluru,55.235786,114.010870,35.472778,6.702294,38.663864,1069.550189,23.4,70,0.0,13.2,913.6
4,2020-01-05,Bengaluru,51.294156,101.563218,30.340575,8.320714,43.419048,688.467578,23.3,73,0.2,16.6,913.4
5,2020-01-06,Bengaluru,22.642570,62.554217,25.856250,5.966067,22.645114,666.959270,22.4,78,0.0,19.0,915.0
6,2020-01-07,Bengaluru,24.838682,61.011628,28.709500,6.703654,29.825634,649.030899,22.0,73,0.0,19.7,914.9
7,2020-01-08,Bengaluru,31.501083,79.282609,33.996337,4.558495,46.321798,800.207985,22.5,67,0.0,16.4,914.1
8,2020-01-09,Bengaluru,41.306190,91.720930,38.944933,6.294483,32.650000,692.804878,22.8,67,0.0,17.9,913.9
9,2020-01-10,Bengaluru,38.605655,89.869048,38.536590,7.940110,33.272222,785.418789,21.7,67,0.0,18.4,913.8


## Validation of the Integrated Dataset

Following the merge, the integrated dataset was validated for structural consistency, duplicate observations, missing values, and city-level coverage.

This validation ensures that the pollution and meteorological records were correctly aligned before feature engineering and predictive modeling.

In [ ]:
print("Number of rows:", len(final_df))
print("Number of columns:", final_df.shape[1])

print("\nRows per city:")
print(final_df["City"].value_counts())

print("\nData types:")
print(final_df.dtypes)

print("\nMissing percentage:")

missing_pct = (
    final_df.isnull().mean() * 100
).round(2)

print(
    missing_pct[
        missing_pct > 0
    ].sort_values(ascending=False)
)

Number of rows: 3655
Number of columns: 13

Rows per city:
City
Bengaluru    731
Chennai      731
Delhi        731
Kolkata      731
Mumbai       731
Name: count, dtype: int64

Data types:
Date                      datetime64[ns]
City                              object
PM2.5                            float64
PM10                             float64
NO2                              float64
SO2                              float64
O3                               float64
CO                               float64
Temperature_Mean                 float64
Relative_Humidity_Mean             int64
Precipitation                    float64
Wind_Speed_Max                   float64
Surface_Pressure_Mean            float64
dtype: object

Missing percentage:
PM10     9.06
SO2      7.55
NO2      7.52
O3       7.50
CO       7.50
PM2.5    4.16
dtype: float64


In [ ]:
# Save integrated cleaned dataset before feature engineering

final_df.to_csv(
    "air_pollution_weather_cleaned_2020_2021.csv",
    index=False
)

print("Saved successfully!")
print("Shape:", final_df.shape)

Saved successfully!
Shape: (3655, 13)


In [ ]:
from google.colab import files

files.download(
    "air_pollution_weather_cleaned_2020_2021.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>